# Run a trained Relational Transformer on your own database

Load a released **RT-J** checkpoint from HuggingFace and predict tasks you define over your
own relational database — here a small **DuckDB** store. The prediction itself is just the
model's forward pass over a sampled context; mapping outputs back to your rows and scoring
are separate steps, shown explicitly below.

*Runtime → Change runtime type → **GPU**; Python **3.12+**.*

## 1. Install (one-time)

In [ ]:
import os, sys
assert sys.version_info >= (3, 12), f"need Python >= 3.12, found {sys.version.split()[0]}"
REPO_URL, BRANCH = "https://github.com/stanford-star/relational-transformer.git", "main"
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y -q
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
![ -d relational-transformer ] || git clone -q -b {BRANCH} {REPO_URL}
%cd relational-transformer
os.environ["PYTHONPATH"], os.environ["MATURIN_IMPORT_HOOK_ENABLED"] = os.getcwd(), "0"
!pip install -q duckdb einops safetensors "huggingface_hub>=0.23" sentence-transformers ml_dtypes orjson strictfire lazy_loader maturin maturin_import_hook
!pip install -q ./rustler                                          # the Sampler extension
!cargo build -q --release --features pre --manifest-path rustler/Cargo.toml   # the `pre` binary
print("setup done")

## 2. Your database, and the tasks you want to predict

`mini-shop.duckdb` is a toy store: `customers`, `products`, and a time-stamped
`transactions` log. A **task** is a point-in-time SQL query emitting `(entity, time, label)`
— the label computed from data *after* the time, for entities with history *before* it.
Swap this cell for your own database and task SQL; nothing else changes.

In [2]:
import duckdb
con = duckdb.connect("examples/byod/mini-shop.duckdb", read_only=True)
con.execute("""
CREATE OR REPLACE TEMP TABLE cutoffs AS                       -- monthly reference dates
  SELECT TIMESTAMP '2020-01-01' + (d * INTERVAL 1 DAY) AS cutoff FROM range(360, 692, 28) t(d);
CREATE OR REPLACE TEMP TABLE eligible AS                      -- customers with history before the cutoff
  SELECT DISTINCT t.customer_id, k.cutoff FROM transactions t JOIN cutoffs k ON t.timestamp < k.cutoff;
CREATE OR REPLACE TEMP TABLE customer_churn AS               -- clf: no purchase in the next 28 days?
  SELECT e.customer_id, e.cutoff AS timestamp, count(t.transaction_id) = 0 AS churn FROM eligible e
  LEFT JOIN transactions t ON t.customer_id = e.customer_id
       AND t.timestamp > e.cutoff AND t.timestamp <= e.cutoff + INTERVAL 28 DAY
  GROUP BY e.customer_id, e.cutoff;
CREATE OR REPLACE TEMP TABLE customer_spend AS              -- reg: total spend in the next 28 days
  SELECT e.customer_id, e.cutoff AS timestamp, COALESCE(sum(t.amount), 0.0) AS spend FROM eligible e
  LEFT JOIN transactions t ON t.customer_id = e.customer_id
       AND t.timestamp > e.cutoff AND t.timestamp <= e.cutoff + INTERVAL 28 DAY
  GROUP BY e.customer_id, e.cutoff;
""")
con.sql("SELECT * FROM customer_churn ORDER BY customer_id, timestamp LIMIT 5").df()

,customer_id,timestamp,churn
0,0,2020-12-26,True
1,0,2021-01-23,True
2,0,2021-02-20,True
3,0,2021-03-20,True
4,0,2021-04-17,True


### Reading from a live SQL database (MySQL / Postgres)

Your data is probably in MySQL or Postgres, not a DuckDB file — and you don't need to change the steps below. DuckDB reads both directly, so swap the `duckdb.connect("…")` above for an `ATTACH` and reference your own tables in the task SQL:

```python
con = duckdb.connect()                                   # in-memory
con.execute("INSTALL mysql; LOAD mysql")                 # Postgres: INSTALL postgres; LOAD postgres
con.execute("ATTACH 'host=… user=… password=… database=shop' AS shop (TYPE mysql)")
#                                                          Postgres: 'dbname=shop …' (TYPE postgres)
# then reference shop.customers, shop.transactions, … in the task SQL and the COPY … TO parquet
```

Everything from step 3 on (export → preprocess → predict) is unchanged — RT only ever sees the exported parquet, never your database.

## 3. Export to RT's input format and preprocess

RT reads a relbench-3.0.0 directory (`manifest.yaml` + `db/*.parquet` +
`tasks/<task>/{train,val,test}.parquet`); DuckDB writes the parquet, split by cutoff (last =
test, prev = val, rest = train). `preprocess` then builds the on-disk format RT samples from.
*(Mechanical — copy as-is.)*

In [3]:
import os, yaml
R = "byod/mini-shop"; os.makedirs(f"{R}/db", exist_ok=True)
for t in ["customers", "products", "transactions"]:
    con.execute(f"COPY {t} TO '{R}/db/{t}.parquet' (FORMAT PARQUET)")
yaml.safe_dump({"name": "mini-shop", "tables": {
    "customers": {"pkey": "customer_id"}, "products": {"pkey": "product_id"},
    "transactions": {"pkey": "transaction_id", "time_col": "timestamp",
                     "fkeys": {"customer_id": "customers", "product_id": "products"}}}},
    open(f"{R}/manifest.yaml", "w"), sort_keys=False)
cuts = [r[0] for r in con.execute("SELECT cutoff FROM cutoffs ORDER BY cutoff").fetchall()]
for tbl, name, col, kind in [("customer_churn", "customer-churn", "churn", "binary_classification"),
                             ("customer_spend", "customer-spend", "spend", "regression")]:
    d = f"{R}/tasks/{name}"; os.makedirs(d, exist_ok=True)
    con.execute(f"COPY (FROM {tbl} WHERE timestamp < ?) TO '{d}/train.parquet' (FORMAT PARQUET)", [cuts[-2]])
    con.execute(f"COPY (FROM {tbl} WHERE timestamp = ?) TO '{d}/val.parquet'   (FORMAT PARQUET)", [cuts[-2]])
    con.execute(f"COPY (FROM {tbl} WHERE timestamp = ?) TO '{d}/test.parquet'  (FORMAT PARQUET)", [cuts[-1]])
    yaml.safe_dump({"entity_table": "customers", "entity_col": "customer_id", "target_col": col,
                    "task_type": kind, "time_col": "timestamp"}, open(f"{d}/manifest.yaml", "w"), sort_keys=False)

!python scripts/preprocess.py one --dataset byod/mini-shop --out-dir byod/pre

## 4. Run the model — a forward pass over sampled contexts

RT is an in-context learner, so a prediction needs a *context*: for each test row,
`build_evaluator` samples one (similar labeled rows + the target masked), and
`evaluate_raw` runs the model's forward over it. That forward **is** the prediction — here we
just collect the raw per-row output and each row's seed node index. The checkpoint's
`config.json` says clf or reg, so only the matching tasks run.

> The `stanford-star/rt-j/classification` and `stanford-star/rt-j/regression` repos must be reachable; if private, run
> `from huggingface_hub import notebook_login; notebook_login()` first. A locally trained
> `best_clf.safetensors` works as the checkpoint too.

In [4]:
import torch
from rt.checkpoints import load_rt_model
from rt.eval_utils import build_evaluator      # samples contexts + drives the model (the data loader)
from rt.tasks import tasks_from_preprocessed

raw = {}  # task_name -> (raw model output, seed node indices, task, config)
for ckpt in ["stanford-star/rt-j/classification", "stanford-star/rt-j/regression"]:
    model, cfg = load_rt_model(ckpt, device="cuda")
    model = model.to(torch.bfloat16)            # the model runs in bf16 (matches the sampled data)
    tasks = [t for t in tasks_from_preprocessed("byod/pre", splits=("test",), dbs=["mini-shop"])
             if t.task_type == cfg["task_type"]]
    ev = build_evaluator(tasks, "byod/pre", embedding_model=cfg["embedding_model"],
                         d_text=cfg["d_text"], device="cuda", ctx_size=8192, items_per_task=10_000_000)
    for task, _ctx, _labels, out, _n, node_idxs in ev.evaluate_raw([(model, "")], [8192], with_node_idxs=True):
        raw[task.table_name] = (out[""], node_idxs, task, cfg)
print("ran the model on:", list(raw))


eval data loaded in 0m00s


  prefetch init took 0m00s



eval data loaded in 0m00s


  prefetch init took 0m00s


ran the model on: ['customer-churn', 'customer-spend']


## 5. Map the outputs back to your rows

The model returns one number per test row, in sampling order; we line each up with its
test-table row by seed node index, then put it on the natural scale — sigmoid for the
classifier, denormalize with the train-split mean/std for the regressor.

In [5]:
import json, numpy as np, pandas as pd
table_info = json.load(open("byod/pre/mini-shop/table_info.json"))
preds = {}
for name, (out, node_idxs, task, cfg) in raw.items():
    offset = table_info[f"{name}:Test"]["node_idx_offset"]
    rows = np.asarray(node_idxs) - offset                       # seed node index -> test-table row
    df = pd.read_parquet(f"byod/mini-shop/tasks/{name}/test.parquet").iloc[rows].reset_index(drop=True)
    out = np.asarray(out, dtype=float)
    if task.task_type == "clf":
        df["prediction"] = 1 / (1 + np.exp(-out))               # logit -> probability
    else:
        train = pd.read_parquet(f"byod/mini-shop/tasks/{name}/train.parquet")[task.target_column]
        df["prediction"] = out * train.std(ddof=1) + train.mean()   # denormalize
    preds[name] = df

display(preds["customer-churn"].head())
display(preds["customer-spend"].head())

,customer_id,timestamp,churn,prediction
0,1487,2021-10-30,True,0.443600
1,1121,2021-10-30,False,0.414899
2,1356,2021-10-30,False,0.447702
3,39,2021-10-30,False,0.403097
4,882,2021-10-30,True,0.431584


,customer_id,timestamp,spend,prediction
0,635,2021-10-30,0.00,43.838799
1,106,2021-10-30,0.00,20.728582
2,128,2021-10-30,69.47,48.584647
3,413,2021-10-30,0.00,37.235880
4,998,2021-10-30,0.00,32.077350


## 6. Score — a separate step on top of the outputs

Metrics are just a function of `(label, prediction)`, independent of the model.

In [6]:
from sklearn.metrics import roc_auc_score, mean_absolute_error
c = preds["customer-churn"]; print(f"customer-churn  AUROC = {roc_auc_score(c.churn, c.prediction):.4f}")
s = preds["customer-spend"]; print(f"customer-spend  MAE   = {mean_absolute_error(s.spend, s.prediction):.4f}")

customer-churn  AUROC = 0.8107
customer-spend  MAE   = 58.5143


## Use your own data

Edit **step 2**: point `duckdb.connect` at your own database (or
`CREATE TABLE ... AS SELECT * FROM read_parquet('...')` / `read_csv('...')`) and write task
SQL emitting `(entity_id, timestamp, label)` — label from data strictly *after* the
timestamp, rows restricted to entities with history *before* it. Give each task a
`tasks/<task>/manifest.yaml` (`entity_table`, `entity_col`, `target_col`, `task_type` =
`binary_classification` | `regression`, `time_col`) in step 3. Foreign keys must be 0-based
parent row indices and time columns real datetimes (already true above). Steps 4–6 are
unchanged and work for any number of tasks.